# Thành viên 2
Các bảng phụ trách: `products.csv`, `reviews.csv`, `customers.csv`, `web_traffic.csv`

Chạy lần lượt từ **Ô 0** đến ô cuối. Mỗi ô tạo đúng một file CSV, đọc từ dữ liệu gốc `student_data`.

## Ô 0: Chuẩn bị

In [1]:
from pathlib import Path
import zipfile
import pandas as pd

DATA = Path("/content/student_data")   # dữ liệu gốc (bronze)
OUT = Path("/content/silver")          # kết quả (silver)
OUT.mkdir(exist_ok=True)

if not DATA.exists():                  # chưa có dữ liệu -> chọn file student_data.zip
    from google.colab import files
    for name in files.upload():
        zipfile.ZipFile(name).extractall("/content")

def strip(df, cols):
    """Bỏ khoảng trắng thừa ở các cột chữ."""
    for c in cols:
        df[c] = df[c].astype("string").str.strip()
    return df

def save(df, name, pk):
    """Kiểm tra khóa chính (duy nhất, không rỗng) rồi ghi CSV."""
    pk = [pk] if isinstance(pk, str) else pk
    assert df[pk].notna().all().all() and not df.duplicated(pk).any(), f"{name}: khóa chính lỗi"
    df.to_csv(OUT / f"{name}.csv", index=False, encoding="utf-8-sig")
    print(f"{name}.csv: {len(df):,} dòng, khóa chính {pk} hợp lệ")
    return df.head()

Saving student_data.zip to student_data.zip


## Ô 1: `products.csv`

In [2]:
products = strip(pd.read_csv(DATA / "products.csv"), ["product_name", "category", "segment", "size", "color"])

save(products.drop_duplicates("product_id"), "products", "product_id")

products.csv: 2,412 dòng, khóa chính ['product_id'] hợp lệ


,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406


## Ô 2: `reviews.csv`

In [3]:
# Sửa: bỏ customer_id (review -> order -> customer, phụ thuộc bắc cầu). Kiểm tra trước khi bỏ
reviews = pd.read_csv(DATA / "reviews.csv")
orders = pd.read_csv(DATA / "orders_enriched.csv", usecols=["order_id", "customer_id"])
chk = reviews.merge(orders, on="order_id", suffixes=("", "_order"))
assert len(chk) == len(reviews) and (chk["customer_id"] == chk["customer_id_order"]).all(), \
    "customer_id của review khác khách của đơn"

reviews["review_date"] = pd.to_datetime(reviews["review_date"])
strip(reviews, ["review_title"])

save(reviews.drop(columns="customer_id").drop_duplicates("review_id"), "reviews", "review_id")

reviews.csv: 113,551 dòng, khóa chính ['review_id'] hợp lệ


,review_id,order_id,product_id,review_date,rating,review_title
0,REV-0000001,1,2400,2012-07-24,5,Highly recommend
1,REV-0000002,3,396,2012-08-03,5,Very satisfied
2,REV-0000003,10,1431,2012-07-23,5,Great quality
3,REV-0000005,16,1668,2012-08-05,5,Great quality
4,REV-0000006,17,2352,2012-07-17,4,Good overall


## Ô 3: `customers.csv`

In [4]:
# Sửa: bỏ city (suy ra từ zip qua bảng location). Kiểm tra city khớp với geography trước khi bỏ
customers = pd.read_csv(DATA / "customers.csv")
geo = pd.read_csv(DATA / "geography.csv", usecols=["zip", "city"])
chk = customers.merge(geo, on="zip", how="left", suffixes=("", "_geo"))
assert chk["city_geo"].notna().all() and (chk["city"] == chk["city_geo"]).all(), "city không khớp zip"

customers["signup_date"] = pd.to_datetime(customers["signup_date"])
strip(customers, ["gender", "age_group", "acquisition_channel"])

save(customers.drop(columns="city").drop_duplicates("customer_id"), "customers", "customer_id")

customers.csv: 121,930 dòng, khóa chính ['customer_id'] hợp lệ


,customer_id,zip,signup_date,gender,age_group,acquisition_channel
0,1,15201,2021-12-30,Female,35-44,social_media
1,2,15201,2013-12-27,Female,45-54,email_campaign
2,3,15201,2018-07-24,Female,18-24,organic_search
3,4,15201,2017-11-29,Male,35-44,referral
4,5,15201,2022-09-23,Male,55+,organic_search


## Ô 4: `web_traffic.csv`

In [5]:
web_traffic = pd.read_csv(DATA / "web_traffic.csv")
web_traffic["date"] = pd.to_datetime(web_traffic["date"])
strip(web_traffic, ["traffic_source"])

save(web_traffic.drop_duplicates(), "web_traffic", "date")

web_traffic.csv: 3,652 dòng, khóa chính ['date'] hợp lệ


,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,2013-01-01,9760,7253,39093,0.00514,102.9,organic_search
1,2013-01-02,10456,8151,47611,0.00406,120.5,organic_search
2,2013-01-03,10076,7458,36963,0.00401,263.6,direct
3,2013-01-04,9973,8063,53078,0.00562,151.8,direct
4,2013-01-05,10223,7882,36790,0.00525,168.6,referral
